In [ ]:
import os
import sys

import importlib
import glob
import copy
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

from myPythonTools.TopoAna import rdridgelist as RL
from myPythonTools.TopoAna import rdremap as RM

from scipy.ndimage import gaussian_filter

def ridge_scale_map(Z, sigmas_km, dx_km=3.336, gamma=0.75):
    """
    Z : 2-D field on one cube face, indexed [y, x] (e.g. dev[p]).
    Returns, per pixel, at the scale with the strongest normalized ridge response:
      sig_best : that sigma (km); for a Gaussian ridge it equals the ridge's own sigma
      R_best   : the normalized ridge strength there
      th_cross : cross-ridge direction, degrees from +x toward +y, in [0, 180)
    """
    R_best   = np.full(Z.shape, -np.inf)
    sig_best = np.full(Z.shape, np.nan)
    th_cross = np.full(Z.shape, np.nan)
    for s_km in sigmas_km:
        s = s_km / dx_km                               # sigma in cells
        Lyy = gaussian_filter(Z, s, order=(2, 0))      # d2/dy2  (axis 0 = y)
        Lxx = gaussian_filter(Z, s, order=(0, 2))      # d2/dx2  (axis 1 = x)
        Lxy = gaussian_filter(Z, s, order=(1, 1))      # d2/dxdy
        half_tr = 0.5 * (Lxx + Lyy)
        rad     = np.sqrt(0.25 * (Lxx - Lyy)**2 + Lxy**2)
        lam_q   = half_tr - rad                        # most negative curvature (across crest)
        lam_p   = half_tr + rad                        # the other one (along crest)
        R = (s**2)**gamma * (-lam_q)                   # gamma-normalized ridge strength
        ridge_like = (lam_q < 0) & (np.abs(lam_q) > np.abs(lam_p))   # a ridge, not a bowl or a cone
        R = np.where(ridge_like, R, -np.inf)
        better = R > R_best
        R_best   = np.where(better, R, R_best)
        sig_best = np.where(better, s_km, sig_best)
        th_p = 0.5 * np.degrees(np.arctan2(2 * Lxy, Lxx - Lyy))    # along-crest eigenvector
        th_cross = np.where(better, (th_p + 90.) % 180., th_cross)
    return sig_best, R_best, th_cross

In [ ]:

TopoRoot = '/glade/work/juliob/Topo/NCARTopoJTB-fire/'     #cases/ne30pg3_Sco100_small_rdgs/
Case = 'ne30pg3_Sco100_Rw050_v205_new_wedge_e03'
ogrid= 'ne30pg3'


foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/topo*.nc')
FileN = foo[-1]
print(FileN)
topo=xr.open_dataset( FileN )
print(list(topo.variables))
print(topo.attrs)
raw=topo['terr_dev']+topo['terr_sm']
Loo = np.size(raw)
nc=int(np.sqrt( Loo/6 ))
print(int(nc))
raw=np.reshape( raw.values , (6,nc,nc) )
dev=np.reshape( topo['terr_dev'].values , (6,nc,nc) )
smoo=np.reshape( topo['terr_sm'].values , (6,nc,nc) )
tlats=np.reshape( topo['lat'].values , (6,nc,nc) )
tlons=np.reshape( topo['lon'].values , (6,nc,nc) )


In [ ]:
foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/remap*dat')
RmpFile = foo[-1]
print(RmpFile)
Rmp=RM.rdremap( RmpFile,  fields="all" )
hwdth=Rmp.hwdth


In [ ]:
from myPythonTools.TopoAna import cube_panels as CP


p = 3                                   # face 4  (0-based)
m = CP.in_box(tlons[p], tlats[p], [268., 285.], [30., 41.])
m = CP.in_box(tlons[p], tlats[p], [268.-30, 285.-30], [30., 41.])
m = CP.in_box(tlons[p], tlats[p], [272.-30, 285.-30], [35., 41.])
#m = CP.in_box(tlons[p], tlats[p], [220., 300], [-20., 42.])
yy, xx = np.nonzero(m)
pad = 60                                # margin in cells (about 4 x the largest sigma)
y0, y1 = max(yy.min()-pad, 0), min(yy.max()+pad+1, 3000)
x0, x1 = max(xx.min()-pad, 0), min(xx.max()+pad+1, 3000)
Z = dev[p, y0:y1, x0:x1]
WW = hwdth[p, y0:y1, x0:x1]
print(Z.shape)

In [ ]:
plt.contourf( Z )

In [ ]:
sigmas_km = [6, 9, 12, 18, 25, 37, 50, 75, 100]
sig_best, R_best, th_cross = ridge_scale_map(Z, sigmas_km,gamma=1.0)

In [ ]:
good   = np.isfinite(R_best)
strong = good & (R_best > np.percentile(R_best[good], 0))
fig, ax = plt.subplots(1, 3, figsize=(30, 8))
co=ax[0].imshow(Z, origin='lower', cmap='terrain', vmin=-1500, vmax=1500)
ax[0].set_title('dev')
plt.colorbar( co , shrink=0.5)
im = ax[1].imshow(np.where(strong, sig_best, np.nan), origin='lower', cmap='viridis')
plt.colorbar(im, ax=ax[1], label='best ridge sigma (km)',shrink=.5)
ax[1].set_title('dominant ridge scale')

wim = ax[2].imshow(WW*3.3, origin='lower',cmap='viridis')
plt.colorbar(wim, ax=ax[2], label='hwdth(km)',shrink=.5)
ax[1].set_title('dominant ridge scale')


In [ ]:
# Do a file search in output directory
# Pick last (most recent) file
foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/Ridge_list*dat')
ListFile = foo[-1]
List=RL.rdridgelist( ListFile )

sel = (List.MyPanel == p+1) & (List.mxdis >= 1)
ix  = np.rint(List.xspk[sel]).astype(int) - 1 - x0     # Fortran 1-based -> cutout index
iy  = np.rint(List.yspk[sel]).astype(int) - 1 - y0
ok  = (ix >= 0) & (ix < Z.shape[1]) & (iy >= 0) & (iy < Z.shape[0])
plt.figure()
plt.scatter(List.hwdth[sel][ok] * 3.336, sig_best[iy[ok], ix[ok]], s=2, alpha=0.3)
plt.xlabel('hwdth from Ridge_list (km)'); plt.ylabel('best sigma here (km)')

In [ ]:
from scipy.ndimage import maximum_filter

# 1) robust sampling: sigma at the strongest-R pixel within +/-3 cells of the crest point
r = 3
Rm = np.where(np.isfinite(R_best), R_best, -np.inf)
iy_c, ix_c = iy[ok], ix[ok]
sig_s = np.full(iy_c.size, np.nan); R_s = np.full(iy_c.size, np.nan)
for k, (a, b) in enumerate(zip(iy_c, ix_c)):
    win = Rm[max(a-r,0):a+r+1, max(b-r,0):b+r+1]
    j = np.unravel_index(np.argmax(win), win.shape)
    sig_s[k] = sig_best[max(a-r,0)+j[0], max(b-r,0)+j[1]]
    R_s[k]   = win[j]

# 2) keep clear ridges only
hw  = List.hwdth[sel][ok] * 3.336
mx  = List.mxdis[sel][ok]
keep = np.isfinite(R_s) & (R_s > np.percentile(R_s[np.isfinite(R_s)], 50)) & (mx > 100.)

# 3) median and quartiles of sigma per hwdth bin, plus counts
bins = np.arange(0, 95, 10)
idx  = np.digitize(hw[keep], bins)
for k in range(1, len(bins)):
    s = sig_s[keep][idx == k]
    if s.size > 20:
        q1, q2, q3 = np.percentile(s, [25, 50, 75])
        print(f"hwdth {bins[k-1]:3d}-{bins[k]:3d} km  n={s.size:5d}  sigma median {q2:5.1f}  IQR {q1:5.1f}-{q3:5.1f}")

In [ ]:
plt.plot( sig_best[280,:]  )

In [ ]:
boo=np.where(sig_best==np.nan)
sig_best[ np.isnan(sig_best) ]=300.
plt.contourf(sig_best,extend='both',levels=np.linspace(0,100,num=31))
plt.colorbar()

In [ ]:
print( np.sum(np.isnan(sig_best)) )

In [ ]:
plt.contourf(1./(sig_best+1.) ,extend='both',levels=31 ) #np.linspace(0,100,num=31))
plt.colorbar()